# Grid Search XGBoost para Dataset Adidas

Esta libreta genera múltiples configuraciones `.yaml` en la carpeta compatible con Hydra:
`hfedxgboost/hfedxgboost/conf/xgboost_params_centralized`

Cada configuración se usa para lanzar el script centralizado y evaluar el rendimiento.
Los resultados se mostrarán por pantalla y pueden ser capturados luego en CSV si se desea.

In [ ]:
!pip install poetry
!poetry install



In [ ]:
import subprocess
from pathlib import Path
import yaml
import itertools
import sys
import pandas as pd

## Preparar datasets

In [ ]:
import subprocess
from pathlib import Path
import sys

prepare_dir = Path("hfedxgboost")
python_path = Path(sys.executable)  # esto apunta a tu Python actual en la libreta

result = subprocess.run(
    [str(python_path), "prepare_adidas.py"],
    cwd=prepare_dir,
    capture_output=True,
    text=True
)

print("📤 STDOUT:\n", result.stdout)
print("\n💥 STDERR:\n", result.stderr)

if result.returncode != 0:
    raise RuntimeError(f"❌ El script falló con código {result.returncode}")




## Ejecución centralizada

In [26]:
# Carpeta donde Hydra busca los .yaml (debe existir)
base_dir = Path("hfedxgboost/conf/xgboost_params_centralized")

base_dir.mkdir(parents=True, exist_ok=True)

In [27]:
# Grid de hiperparámetros 
param_grid = {
    'n_estimators': [500, 800],
    'max_depth': [8, 10, 12, 15],               # más profundidad
    'subsample': [0.7, 0.9, 1.0],               # valores altos
    'learning_rate': [0.1, 0.05, 0.03],         # bajamos más
    'colsample_bylevel': [1],                   # lo mantenemos fijo
    'colsample_bynode': [1],
    'colsample_bytree': [1],
    'alpha': [0],                               # lo eliminamos (no penalización)
    'gamma': [0, 1],                            # muy bajo (menos penalización)
    'num_parallel_tree': [1],
    'min_child_weight': [1, 3],                 # menor peso mínimo
}



combinaciones = list(itertools.product(*param_grid.values()))
keys = list(param_grid.keys())

for i, values in enumerate(combinaciones):
    config = {k: v for k, v in zip(keys, values)}
    yaml_name = f"adidas_xgb_run_{i:03d}"
    yaml_path = base_dir / f"{yaml_name}.yaml"

    with open(yaml_path, "w") as f:
        yaml.dump(config, f, sort_keys=False)  

    print(f"✔️ Generado: {yaml_path}")

    # Ejecutar el experimento
    print(f"🚀 Ejecutando experimento {i+1}/{len(combinaciones)}...")
    result = subprocess.run([
        str(Path(sys.executable)), "-m", "hfedxgboost.main",
        "--config-name", "Centralized_Baseline",
        "dataset=adidas",
        f"xgboost_params_centralized={yaml_name}"
    ], capture_output=True, text=True)

    print("📤 STDOUT:")
    print(result.stdout)
    print("💥 STDERR:")
    print(result.stderr)



✔️ Generado: hfedxgboost\conf\xgboost_params_centralized\adidas_xgb_run_000.yaml
🚀 Ejecutando experimento 1/288...
📤 STDOUT:
dataset:
  task:
    task_type: REG
    metric:
      name: mse
      fn:
        _target_: torchmetrics.MeanSquaredError
    criterion:
      _target_: torch.nn.MSELoss
    xgb:
      _target_: xgboost.XGBRegressor
      objective: reg:squarederror
  dataset_name: adidas
  train_ratio: 0.8
  early_stop_patience_rounds: 5
xgboost_params_centralized:
  n_estimators: 500
  max_depth: 8
  subsample: 0.7
  learning_rate: 0.1
  colsample_bylevel: 1
  colsample_bynode: 1
  colsample_bytree: 1
  alpha: 0
  gamma: 0
  num_parallel_tree: 1
  min_child_weight: 1
centralized: true
n_estimators_client: ${xgboost_params_centralized.n_estimators}
task_type: ${dataset.task.task_type}
XGBoost:
  _target_: ${dataset.task.xgb._target_}
  objective: ${dataset.task.xgb.objective}
  learning_rate: ${xgboost_params_centralized.learning_rate}
  max_depth: ${xgboost_params_centralized.m

KeyboardInterrupt: 

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Cargar el CSV de resultados
df = pd.read_csv("results_centralized.csv")

# Ordenar por menor error de test
df_sorted = df.sort_values(by="result_test", ascending=True)

# Mostrar las mejores y peores
print("🏆 Mejores configuraciones:")
display(df_sorted.head(5))

print("💥 Peores configuraciones:")
display(df_sorted.tail(5))

# Plot evolución del error
plt.figure(figsize=(10, 4))
plt.plot(df_sorted["result_test"].values, marker='o')
plt.title("Error de Test por configuración (ordenado)")
plt.ylabel("MSE")
plt.xlabel("Configuración")
plt.grid(True)
plt.show()

# Heatmap de correlación entre hiperparámetros y resultado
param_cols = [
    'xgb_max_depth', 'subsample', 'learning_rate',
    'colsample_bylevel', 'colsample_bynode', 'colsample_bytree',
    'alpha', 'gamma', 'num_parallel_tree', 'min_child_weight'
]

# Crear matriz de correlación
corr_matrix = df[param_cols + ['result_test']].corr()

# Visualizar heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix[["result_test"]].sort_values(by="result_test", ascending=False), 
            annot=True, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlación de hiperparámetros con el error de test")
plt.show()


## Ejecución federada